**UNIVERSIDAD NACIONAL DEL CHIMBORAZO**

**NOMBRE:** Cristian Alvear

**CURSO:** Segundo A

**FECHA:** 28/04/2026

**TEMA:**

**ACTIVIDAD AUTÓNOMA 02** 


Contexto de la tarea:
Un asistente administrativo guardó la información de los consultores de la empresa en una
nota de texto manual, cometiendo múltiples errores: mezcló separadores (;, ,, -), escribió
nombres de forma inconsistente y dejó "ruido" (caracteres extraños) al final de los registros.
Tu misión: Mañana es el día de pago. Debes actuar como el programador encargado de
limpiar este desastre. Tu script debe procesar ese texto sucio y convertirlo en una lista de
registros (diccionarios) organizada para que el sistema pueda calcular los sueldos y
realizar las transferencias a tiempo.

In [21]:
data_set = [
    "ID:001; NAME: alberto_EINSTEIN ; PROYECTO:Relatividad; HORAS:40; PAGO_H:35.5",
    "id-002, name- marie_CURIE, proyecto-Radiactividad, horas-35, pago_h-40.0",
    "ID:003; NAME:isaac_NEWTON; PROYECTO:Leyes del Movimiento; HORAS:20; PAGO_H:30.0####",
    " id:004 - name: nikola_TESLA - proyecto:Corriente Alterna - horas:50 - pago_h:25.0",
    "ID:005; NAME:ada_LOVELACE; PROYECTO:Algoritmos; HORAS:45; PAGO_H:45.0",
    "id-006, name- charles_DARWIN, proyecto-Evolución, horas-30, pago_h-20.0????",
    "ID:007; NAME: rosalind_FRANKLIN ; PROYECTO:Estructura ADN; HORAS:38; PAGO_H:32.5",
    "id:008 - name: stephen_HAWKING - proyecto:Agujeros Negros - horas:15 - pago_h:50.0",
    "ID:009; NAME:katherine_JOHNSON; PROYECTO:Trayectorias NASA; HORAS:42; PAGO_H:40.0!!!!",
    "id-010, name- grace_HOPPER, proyecto-Compiladores, horas-48, pago_h-38.0"
]

# PARTE 1: Limpieza de Datos

## Estrategia de Limpieza

Voy a usar **expresiones regulares (regex)** porque:

1.  Maneja múltiples separadores en una sola operación
2.  No importa el formato (`:`, `-`, `,`)
3.  Extrae directamente los valores que necesito

##  Esquema Requerido

Cada registro limpio debe tener:

| Campo | Tipo | Formato |
|-------|------|---------|
| `id` | int | Número sin prefijos (ej: 3) |
| `investigador` | str | Nombre Apellido (ej: Isaac Newton) |
| `area` | str | MAYÚSCULAS (ej: RELATIVIDAD) |
| `salario_semanal` | float | horas × pago_h (ej: 600.00) |

In [22]:
def limpiar_dataset(datos_crudos):
    """
    Transforma la lista de strings en una lista de registros (diccionarios)
    con estructura uniforme.
    """
    registros = []  # Arreglo de estructuras (lista de diccionarios)
 
    for linea in datos_crudos:
        # Paso 1: quitar espacios y caracteres especiales al inicio/final
        linea = linea.strip()
        linea = linea.strip("#?!")
        linea = linea.strip()  # por si quedaron espacios después
 
        # Paso 2: NORMALIZAR separadores
        # Reemplazamos ';' y ' - ' y ',' por un separador único '|'
        # Importante: cuidamos el orden para evitar romper guiones internos
        linea = linea.replace(";", "|")
        linea = linea.replace(" - ", "|")
        linea = linea.replace(",", "|")
 
        # Paso 3: separar los campos por '|'
        campos = linea.split("|")
 
        # Paso 4: procesar cada campo (clave:valor o clave-valor)
        registro_temp = {}
        for campo in campos:
            campo = campo.strip()
            if not campo:
                continue
 
            # Normalizar separador clave-valor: tanto ':' como '-' (el primero que aparezca)
            # Buscamos cuál separador aparece primero
            pos_dos_puntos = campo.find(":")
            pos_guion = campo.find("-")
 
            if pos_dos_puntos != -1 and (pos_guion == -1 or pos_dos_puntos < pos_guion):
                clave, valor = campo.split(":", 1)
            elif pos_guion != -1:
                clave, valor = campo.split("-", 1)
            else:
                continue
 
            clave = clave.strip().lower()
            valor = valor.strip()
            registro_temp[clave] = valor
 
        # Paso 5: armar el registro final con las llaves requeridas
        # ID como entero (quitar ceros a la izquierda)
        id_num = int(registro_temp["id"])
 
        # Investigador: quitar guion bajo y poner formato Título
        nombre_crudo = registro_temp["name"].replace("_", " ").strip()
        investigador = nombre_crudo.title()  # "Alberto Einstein"
 
        # Área en mayúsculas
        area = registro_temp["proyecto"].strip().upper()
 
        # Salario semanal = horas * pago_h
        horas = int(registro_temp["horas"])
        pago_h = float(registro_temp["pago_h"])
        salario_semanal = horas * pago_h
 
        # Registro final (estructura tipo struct con campos definidos)
        registro = {
            "id": id_num,
            "investigador": investigador,
            "area": area,
            "horas": horas,            # lo guardo aparte para análisis posterior
            "pago_h": pago_h,
            "salario_semanal": salario_semanal
        }
 
        registros.append(registro)
 
    return registros

# 📊 PARTE 2: Operaciones de Análisis

Ahora que tenemos los datos limpios, vamos a hacer 3 operaciones:

1. 📋 Mostrar todos los investigadores
2. 🏆 Encontrar al mejor pagado
3. 📈 Calcular promedio de horas trabajadas

In [23]:
def mostrar_investigadores(registros):
    """Muestra id, investigador, área y salario semanal en formato tabla."""
    print("\n" + "=" * 80)
    print("LISTADO DE INVESTIGADORES")
    print("=" * 80)
    print(f"{'ID':<4} {'INVESTIGADOR':<22} {'ÁREA':<28} {'SALARIO SEMANAL':>15}")
    print("-" * 80)
    for r in registros:
        print(f"{r['id']:<4} {r['investigador']:<22} {r['area']:<28} ${r['salario_semanal']:>13,.2f}")
    print("=" * 80)
 
 
def investigador_mejor_pagado(registros):
    """Devuelve el investigador con el salario semanal más alto."""
    mejor = registros[0]
    for r in registros[1:]:
        if r["salario_semanal"] > mejor["salario_semanal"]:
            mejor = r
    return mejor
 
 
def promedio_horas(registros):
    """Calcula el promedio de horas trabajadas de todos los registros."""
    total = 0
    for r in registros:
        total += r["horas"]
    return total / len(registros)

# ============================================================
# PARTE 3: FUNCIONALIDAD PROPIA
# Generador de credenciales de acceso al sistema de la institución
# ============================================================

In [24]:
def generar_credenciales(registros):
    """
    Genera una credencial única para cada investigador con el formato:
        <3 primeras letras del apellido en MAYÚSCULA> + <ID con 3 dígitos> + '-' + <inicial del nombre>
 
    Ejemplo: Alberto Einstein con ID 1  ->  EIN001-A
             Marie Curie    con ID 2  ->  CUR002-M
 
    Esta funcionalidad serviría, por ejemplo, para entregarles un usuario
    de acceso al sistema interno de proyectos de la institución.
    """
    print("\n" + "=" * 80)
    print("CREDENCIALES DE ACCESO GENERADAS")
    print("=" * 80)
    print(f"{'INVESTIGADOR':<22} {'CREDENCIAL':<15}")
    print("-" * 80)
 
    credenciales = []
    for r in registros:
        partes = r["investigador"].split()
        nombre = partes[0]
        apellido = partes[-1]
 
        codigo_apellido = apellido[:3].upper()
        id_formateado = f"{r['id']:03d}"          # rellena con ceros: 1 -> "001"
        inicial = nombre[0].upper()
 
        credencial = f"{codigo_apellido}{id_formateado}-{inicial}"
        credenciales.append({"investigador": r["investigador"], "credencial": credencial})
 
        print(f"{r['investigador']:<22} {credencial:<15}")
    print("=" * 80)
    return credenciales

# ============================================================
# PROGRAMA PRINCIPAL
# ============================================================

In [25]:
if __name__ == "__main__":
    # PARTE 1: limpieza
    registros = limpiar_dataset(data_set)
 
    # PARTE 2: análisis
    mostrar_investigadores(registros)
 
    mejor = investigador_mejor_pagado(registros)
    print(f"\n>> Investigador con MAYOR salario semanal:")
    print(f"   {mejor['investigador']} - Área: {mejor['area']} - "
          f"Salario: ${mejor['salario_semanal']:,.2f}")
 
    prom = promedio_horas(registros)
    print(f"\n>> Promedio de horas trabajadas: {prom:.2f} horas")
 
    # PARTE 3: funcionalidad propia
    generar_credenciales(registros)
 


LISTADO DE INVESTIGADORES
ID   INVESTIGADOR           ÁREA                         SALARIO SEMANAL
--------------------------------------------------------------------------------
1    Alberto Einstein       RELATIVIDAD                  $     1,420.00
2    Marie Curie            RADIACTIVIDAD                $     1,400.00
3    Isaac Newton           LEYES DEL MOVIMIENTO         $       600.00
4    Nikola Tesla           CORRIENTE ALTERNA            $     1,250.00
5    Ada Lovelace           ALGORITMOS                   $     2,025.00
6    Charles Darwin         EVOLUCIÓN                    $       600.00
7    Rosalind Franklin      ESTRUCTURA ADN               $     1,235.00
8    Stephen Hawking        AGUJEROS NEGROS              $       750.00
9    Katherine Johnson      TRAYECTORIAS NASA            $     1,680.00
10   Grace Hopper           COMPILADORES                 $     1,824.00

>> Investigador con MAYOR salario semanal:
   Ada Lovelace - Área: ALGORITMOS - Salario: $2,025.00